# Simple Checks of Multiverse Importance Measures

Here, we check that the multiverse importance measures we have implemented yield results consistent with what we expect. We use toy examples, including some from our update slides, where we know the "ground-truth" according to multiple metrics.

In [ ]:
import itertools
import pandas as pd

from src.metrics.metrics import (
    within_bucket_variation,
    organize_multiverse_results,
    total_variance_decomposition_with_interactions,
)

In [ ]:
def make_example_interactions():
    """
    Create a minimal 2×2×2 multiverse result set where all the importance
    is due to interactions.
    """
    data = pd.DataFrame({
        "dim_A": ["a1", "a1", "a1", "a1", "a2", "a2", "a2", "a2"],
        "dim_B": ["b1", "b1", "b2", "b2", "b1", "b1", "b2", "b2"],
        "dim_C": ["c1", "c2", "c1", "c2", "c1", "c2", "c1", "c2"],
        "accuracy": [0.6, 0.7, 0.8, 0.9, 0.9, 0.8, 0.7, 0.6],
    })
    return organize_multiverse_results(
        data,
        dimension_order=["dim_A", "dim_B", "dim_C"],
        performance_metrics=["accuracy"],
    )

In [ ]:
def make_example_one_feature():
    """
    Create a minimal 2×2×2 multiverse result set where all the importance
    is due to a fixed effect from one feature.
    """
    data = pd.DataFrame({
        "dim_A": ["a1", "a1", "a1", "a1", "a2", "a2", "a2", "a2"],
        "dim_B": ["b1", "b1", "b2", "b2", "b1", "b1", "b2", "b2"],
        "dim_C": ["c1", "c2", "c1", "c2", "c1", "c2", "c1", "c2"],
        "accuracy": [0.6, 0.6, 0.6, 0.6, 0.9, 0.9, 0.9, 0.9],
    })
    return organize_multiverse_results(
        data,
        dimension_order=["dim_A", "dim_B", "dim_C"],
        performance_metrics=["accuracy"],
    )

In [ ]:
def make_example_single_interaction():
    """
    Create a minimal 2×2×2 multiverse result set where all the importance
    is due to a single interaction.
    """
    data = pd.DataFrame({
        "dim_A": ["a1", "a1", "a1", "a1", "a2", "a2", "a2", "a2"],
        "dim_B": ["b1", "b1", "b2", "b2", "b1", "b1", "b2", "b2"],
        "dim_C": ["c1", "c2", "c1", "c2", "c1", "c2", "c1", "c2"],
        "dim_D": ["d1", "d1", "d2", "d2", "d1", "d1", "d2", "d2"],
        "dim_E": ["e1", "e1", "e2", "e2", "e1", "e1", "e2", "e2"],
        "dim_F": ["f1", "f1", "f2", "f2", "f1", "f1", "f2", "f2"],
        "dim_G": ["g1", "g1", "g2", "g2", "g1", "g1", "g2", "g2"],
        "dim_H": ["h1", "h1", "h2", "h2", "h1", "h1", "h2", "h2"],
        "dim_I": ["i1", "i1", "i2", "i2", "i1", "i1", "i2", "i2"],
        "dim_J": ["j1", "j1", "j2", "j2", "j1", "j1", "j2", "j2"],
        "dim_K": ["k1", "k1", "k2", "k2", "k1", "k1", "k2", "k2"],
        "dim_L": ["l1", "l1", "l2", "l2", "l1", "l1", "l2", "l2"],

        "accuracy": [0.9, 0.5, 0.9, 0.5, 0.6, 0.8, 0.6, 0.8],
    })
    return organize_multiverse_results(
        data,
        dimension_order=["dim_A", "dim_B", "dim_C", "dim_D", "dim_E", "dim_F", "dim_G", "dim_H", "dim_I", "dim_J", "dim_K", "dim_L"],
        # dimension_order=["dim_A", "dim_C", "dim_B"],
        performance_metrics=["accuracy"],
    )

In [ ]:
def make_example_single_repeated_interaction():
    """
    Create a full factorial 2^6 multiverse result set (64 rows, one per unique
    combination of 6 binary dimensions) where accuracy is strictly a function
    of the dim_A × dim_C interaction:
        A1 C1 → 0.9,  A1 C2 → 0.6,  A2 C1 → 0.7,  A2 C2 → 0.8
    """
    accuracy_map = {
        ("a1", "c1"): 0.9,
        ("a1", "c2"): 0.6,
        ("a2", "c1"): 0.7,
        ("a2", "c2"): 0.8,
    }

    dims    = ["dim_A", "dim_B", "dim_C", "dim_D", "dim_E", "dim_F"]
    letters = ["a",     "b",     "c",     "d",     "e",     "f"    ]

    combos = list(itertools.product(*[[f"{letter}1", f"{letter}2"] for letter in letters]))

    data = {dim: [combo[i] for combo in combos] for i, dim in enumerate(dims)}
    data["accuracy"] = [accuracy_map[(combo[0], combo[2])] for combo in combos]

    return organize_multiverse_results(
        pd.DataFrame(data),
        # dimension_order=dims,
        dimension_order=["dim_C", "dim_B", "dim_A", "dim_D", "dim_E", "dim_F"],
        performance_metrics=["accuracy"],
    )

In [ ]:
def test_within_bucket_variation():
    mv = make_example_single_repeated_interaction()
    result = within_bucket_variation(mv, "accuracy")
    return result

In [ ]:
def test_between_bucket_variation():
    mv = make_example_single_repeated_interaction()
    result = total_variance_decomposition_with_interactions(mv, "accuracy")
    return result

In [ ]:
within_result = test_within_bucket_variation()
decomp_result = test_between_bucket_variation()

In [ ]:
within_result

In [ ]:
for dim, inner_dict in decomp_result.items():
    print(f"{dim}: {inner_dict['expected_var_within_groups']}")